# 📖 MiniGPT · Dom Casmurro — Inferência
### *Geração de texto ao estilo de Machado de Assis*

---

Notebook **autocontido** para carregar o bundle treinado e gerar texto.
Funciona localmente (CPU/GPU) e no Google Colab.

**Pré-requisito:** arquivo `minigpt_domcasmurro_bundle.pt` na mesma pasta
(ou faça upload na célula seguinte se estiver no Colab).

**Dependências:** apenas `torch` — sem mais nada.


In [ ]:
import os, sys, math, time
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── Detectar ambiente ─────────────────────────────────────────────────────────
IS_COLAB  = "google.colab" in sys.modules
IS_KAGGLE = os.path.exists("/kaggle/working")
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {device}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
print(f"Colab    : {IS_COLAB}  |  Kaggle: {IS_KAGGLE}")

# ── Upload do bundle no Colab ─────────────────────────────────────────────────
BUNDLE_PATH = "minigpt_domcasmurro_bundle.pt"   # ajuste se necessário

if IS_COLAB and not os.path.exists(BUNDLE_PATH):
    print("\n📎 Colab detectado — faça upload do bundle:")
    from google.colab import files
    uploaded = files.upload()                     # abre o seletor de arquivo
    BUNDLE_PATH = list(uploaded.keys())[0]
    print(f"✅ Arquivo recebido: {BUNDLE_PATH}")
elif os.path.exists(BUNDLE_PATH):
    size_mb = os.path.getsize(BUNDLE_PATH) / 1e6
    print(f"\n✅ Bundle encontrado: {BUNDLE_PATH}  ({size_mb:.1f} MB)")
else:
    print(f"\n⚠️  '{BUNDLE_PATH}' não encontrado.")
    print("   Coloque o arquivo na mesma pasta deste notebook")
    print("   ou ajuste a variável BUNDLE_PATH acima.")


---
## 🏗️ Arquitetura do Modelo

As classes abaixo são uma **cópia fiel** do notebook de treino.
Precisam estar definidas para que o `state_dict` carregue corretamente.


In [ ]:
# ── Arquitetura MiniGPT (deve ser idêntica ao notebook de treino) ─────────────

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, context_len, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model    = d_model
        self.n_heads    = n_heads
        self.d_head     = d_model // n_heads
        self.qkv_proj   = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj   = nn.Linear(d_model, d_model,     bias=False)
        self.attn_drop  = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(context_len, context_len))
              .view(1, 1, context_len, context_len)
        )

    def forward(self, x):
        B, T, C = x.shape
        qkv     = self.qkv_proj(x)
        Q, K, V = qkv.split(self.d_model, dim=-1)
        def sh(t):
            return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        Q, K, V = sh(Q), sh(K), sh(V)
        scale   = math.sqrt(self.d_head)
        scores  = (Q @ K.transpose(-2, -1)) / scale
        mask    = self.causal_mask[:, :, :T, :T]
        scores  = scores.masked_fill(mask == 0, torch.finfo(scores.dtype).min)
        w       = F.softmax(scores, dim=-1)
        w       = self.attn_drop(w)
        out     = w @ V
        out     = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.out_proj(out))


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, context_len, dropout):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, context_len, dropout)
        self.ln2  = nn.LayerNorm(d_model)
        self.ff   = FeedForward(d_model, d_ff, dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, context_len, d_model,
                 n_heads, n_layers, d_ff, dropout=0.0):
        super().__init__()
        self.context_len = context_len
        self.token_emb   = nn.Embedding(vocab_size, d_model)
        self.pos_emb     = nn.Embedding(context_len, d_model)
        self.drop        = nn.Dropout(dropout)
        self.blocks      = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, context_len, dropout)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.token_emb.weight

    def forward(self, idx, targets=None):
        B, T   = idx.shape
        pos    = torch.arange(T, device=idx.device)
        x      = self.drop(self.token_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        x      = self.ln_f(x)
        logits = self.head(x)
        loss   = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1)
            )
        return logits, loss

    def count_params(self):
        return sum(p.numel() for p in self.parameters())

    @torch.no_grad()
    def generate(self, prompt_ids, tokenizer,
                 max_new_tokens=200, temperature=1.0, top_k=50):
        self.eval()
        idx = prompt_ids.clone()
        for _ in range(max_new_tokens):
            ctx    = idx[:, -self.context_len:]
            logits, _ = self(ctx)
            logits = logits[:, -1, :] / max(temperature, 1e-6)
            v, _   = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float("-inf")
            probs  = F.softmax(logits, dim=-1)
            tok    = torch.multinomial(probs, num_samples=1)
            idx    = torch.cat([idx, tok], dim=1)
        return tokenizer.decode(idx[0].tolist())


print("✅ Arquitetura definida")


---
## 📦 Carregar Bundle

O bundle contém tudo: **pesos**, **configuração** e **vocabulário completo**.
Não é necessário o corpus original nem nenhum outro arquivo.


In [ ]:
class BundleTokenizer:
    """Reconstrói o tokenizador diretamente do vocabulário salvo no bundle."""
    def __init__(self, vocab: dict):
        self.chars       = vocab["chars"]
        self.vocab_size  = len(self.chars)
        self.char_to_idx = vocab["char_to_idx"]
        self.idx_to_char = {int(k): ch for k, ch in vocab["idx_to_char"].items()}

    def encode(self, text: str) -> list:
        return [self.char_to_idx[c] for c in text if c in self.char_to_idx]

    def decode(self, ids) -> str:
        return "".join(self.idx_to_char.get(i, "?") for i in ids)

    def __len__(self):
        return self.vocab_size


# ── Carregar ──────────────────────────────────────────────────────────────────
if not os.path.exists(BUNDLE_PATH):
    raise FileNotFoundError(f"Bundle não encontrado: {BUNDLE_PATH}")

bundle = torch.load(BUNDLE_PATH, map_location=device, weights_only=False)

tokenizer = BundleTokenizer(bundle["vocab"])
cfg       = bundle["config"]
model     = MiniGPT(**cfg, dropout=0.0).to(device)
model.load_state_dict(bundle["model_state"])
model.eval()

# ── Info ──────────────────────────────────────────────────────────────────────
tr   = bundle["training"]
meta = bundle["metadata"]

print("=" * 55)
print("  MODELO CARREGADO")
print("=" * 55)
print(f"  Corpus        : {meta['corpus']}")
print(f"  Treinado em   : {meta['timestamp']}")
print(f"  Ambiente      : {meta.get('ambiente', '—')}")
print(f"  Parâmetros    : {model.count_params():,}")
print(f"  Melhor época  : {tr['best_epoch']}")
print(f"  Melhor val_loss: {tr['best_val']:.4f}")
print(f"  Vocab size    : {tokenizer.vocab_size}")
print(f"  Context len   : {cfg['context_len']}")
print(f"  d_model       : {cfg['d_model']}")
print(f"  n_layers      : {cfg['n_layers']}")
print(f"  n_heads       : {cfg['n_heads']}")
print("=" * 55)


---
## ✍️ Geração de Texto

### Parâmetros principais

| Parâmetro | Efeito | Valor recomendado |
|---|---|---|
| `temperature` | < 1 conservador · > 1 criativo | 0.7 – 0.9 |
| `top_k` | limita o vocabulário ativo | 30 – 50 |
| `max_new_tokens` | comprimento do texto gerado | 150 – 400 |


In [ ]:
def gerar(prompt, max_new_tokens=200, temperature=0.8, top_k=40, verbose=True):
    """
    Gera texto a partir de um prompt.

    Args:
        prompt          : texto inicial
        max_new_tokens  : caracteres a gerar além do prompt
        temperature     : criatividade (0.1 conservador → 1.5 criativo)
        top_k           : filtra os K tokens mais prováveis
        verbose         : exibe separadores e tempo

    Returns:
        str: prompt + texto gerado
    """
    ids = tokenizer.encode(prompt)
    if not ids:
        raise ValueError("Prompt contém apenas caracteres fora do vocabulário.")

    prompt_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

    t0  = time.time()
    out = model.generate(
        prompt_ids, tokenizer,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
    )
    elapsed = time.time() - t0
    tokens_gerados = len(tokenizer.encode(out)) - len(ids)

    if verbose:
        print(f"[τ={temperature}  k={top_k}  {tokens_gerados} tokens  {elapsed:.1f}s]")
        print("─" * 60)
        print(out)
        print("─" * 60)

    return out


# ── Teste rápido ──────────────────────────────────────────────────────────────
_ = gerar("Capitu olhou para mim", max_new_tokens=100, temperature=0.8)


---
## 🧪 Experimentos

### Experimento 1 — Comparar temperaturas

Mesmo prompt, diferentes temperaturas. Observe como o estilo muda.


In [ ]:
prompt = "Era uma vez um homem chamado Bentinho"

print(f'PROMPT: "{prompt}"')
print("=" * 65)

for tau in [0.4, 0.7, 1.0, 1.3]:
    print(f"\n── temperatura = {tau} " + "─" * 40)
    saida = gerar(prompt, max_new_tokens=160, temperature=tau, top_k=40)
    continuacao = saida[len(prompt):]
    print(continuacao[:200])


### Experimento 2 — Prompts variados

Diferentes pontos de partida ativam diferentes "memórias" do corpus.


In [ ]:
prompts = [
    # Frase icônica do livro
    "Capitu era formosa, de olhos de ressaca",
    # Contexto introspectivo
    "Não há tristeza maior do que",
    # Diálogo
    '"Você acredita em mim?" — perguntou ela',
    # Passagem de tempo
    "Muitos anos depois, recordei que",
    # Fora do domínio (teste de generalização)
    "O computador processava os dados com",
]

for p in prompts:
    print(f"\nPROMPT : {p!r}")
    saida = gerar(p, max_new_tokens=180, temperature=0.8, top_k=40, verbose=False)
    continuacao = saida[len(p):]
    print(f"OUTPUT : {continuacao[:200]}")
    print("─" * 60)


### Experimento 3 — Geração longa

Testar coerência em textos mais longos (400+ tokens).


In [ ]:
print("GERAÇÃO LONGA — 400 tokens\n")
print("=" * 65)

saida_longa = gerar(
    "Capitu",
    max_new_tokens = 400,
    temperature    = 0.75,
    top_k          = 45,
    verbose        = False,
)
print(saida_longa)


---
## 📊 Análise do Treinamento

Visualizar o histórico de loss salvo no bundle.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

history = tr["history"]

if history["train_loss"]:
    ep = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # ── Loss ─────────────────────────────────────────────────────────────────
    axes[0].plot(ep, history["train_loss"], label="Train",
                 color="royalblue", linewidth=2)
    axes[0].plot(ep, history["val_loss"],   label="Val",
                 color="tomato",    linewidth=2)

    # Melhor época
    best_ep  = tr["best_epoch"]
    best_val = tr["best_val"]
    axes[0].axvline(x=best_ep, color="gold", linestyle="--",
                    linewidth=1.5, label=f"Melhor época ({best_ep})")
    axes[0].scatter([best_ep], [best_val], color="gold", zorder=5, s=60)
    axes[0].annotate(f"  {best_val:.4f}", (best_ep, best_val),
                     color="gold", fontsize=9)

    axes[0].set_title("Curvas de Loss", fontsize=13)
    axes[0].set_xlabel("Época")
    axes[0].set_ylabel("Cross-Entropy Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # ── Tempo por época ───────────────────────────────────────────────────────
    if history.get("epoch_time"):
        axes[1].bar(ep, [t/60 for t in history["epoch_time"]],
                    color="mediumseagreen", alpha=0.7, edgecolor="white")
        media = np.mean(history["epoch_time"]) / 60
        axes[1].axhline(y=media, color="darkgreen", linestyle="--",
                        label=f"Média: {media:.1f} min/época")
        axes[1].set_title("Tempo por Época", fontsize=13)
        axes[1].set_xlabel("Época")
        axes[1].set_ylabel("Minutos")
        axes[1].legend()
        axes[1].grid(alpha=0.3)

    plt.suptitle(
        f"MiniGPT — {len(ep)} épocas · melhor val_loss = {best_val:.4f}",
        fontsize=12, y=1.02
    )
    plt.tight_layout()
    plt.savefig("training_history.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Resumo numérico
    print(f"\nResumo do treinamento:")
    print(f"  Épocas completadas  : {len(ep)}")
    print(f"  Train loss final    : {history['train_loss'][-1]:.4f}")
    print(f"  Val loss final      : {history['val_loss'][-1]:.4f}")
    print(f"  Melhor val loss     : {best_val:.4f}  (época {best_ep})")
    melhora = 100 * (1 - best_val / history['val_loss'][0])
    print(f"  Melhora total       : {melhora:.1f}%")
    if history.get("epoch_time"):
        total_min = sum(history["epoch_time"]) / 60
        print(f"  Tempo total treino  : {total_min:.0f} min ({total_min/60:.1f}h)")
else:
    print("⚠️  Histórico de loss não disponível neste bundle.")


---
## 🎮 Modo Interativo

Escreva seus próprios prompts abaixo. Ajuste os parâmetros livremente.


In [ ]:
# ── Edite aqui e re-execute a célula ─────────────────────────────────────────

MEU_PROMPT      = "Capitu sorriu e disse"
MEU_TEMPERATURE = 0.8
MEU_TOP_K       = 40
MEU_MAX_TOKENS  = 250

# ─────────────────────────────────────────────────────────────────────────────
resultado = gerar(
    MEU_PROMPT,
    max_new_tokens = MEU_MAX_TOKENS,
    temperature    = MEU_TEMPERATURE,
    top_k          = MEU_TOP_K,
)


---
## 🔤 Análise do Vocabulário

Inspecionar o vocabulário aprendido pelo modelo.


In [ ]:
from collections import Counter

print(f"Tamanho do vocabulário : {tokenizer.vocab_size} caracteres únicos")
print(f"\nVocabulário completo:")
print("".join(tokenizer.chars))

# Frequência de uso dos tokens num texto gerado
texto_amostra = gerar(
    "Era uma vez",
    max_new_tokens=500, temperature=0.8, top_k=40, verbose=False
)
freq = Counter(texto_amostra)
print(f"\nTop 20 caracteres mais usados na geração:")
for ch, cnt in freq.most_common(20):
    barra = "█" * (cnt // 5)
    print(f"  {repr(ch):6s} : {cnt:4d}  {barra}")


---
## 🚀 Próximos Passos

### Continuar o treinamento

Se quiser treinar mais épocas partindo deste bundle:

```python
# No notebook de treino, carregue o bundle como ponto de partida
model_inf, tokenizer_inf, info = load_bundle("minigpt_domcasmurro_bundle.pt")
# Depois ajuste MAX_EPOCHS e execute treinar() com resume=False
# (o bundle não tem estado do optimizer, só os pesos)
```

### Rodar o app Streamlit

```bash
pip install streamlit torch
streamlit run app.py
# Faça upload do bundle pelo painel lateral do app
```

### Experimentos sugeridos

- Temperatura `0.3` → texto mais "seguro" e repetitivo
- Temperatura `1.4` → texto mais ousado, pode "delirar"
- Prompt em inglês → o modelo ainda gera em português (por quê?)
- Prompt com nome de personagem de outro livro → como o modelo reage?

---
*MiniGPT · Transformer Decoder-Only · ~4.8M parâmetros · Dom Casmurro (Machado de Assis, 1899)*
